알츠하이머 분류를 위한 최종 정형 데이터인 hippo_features_clean_icv.csv 파일 필요함

추가적으로 10개의 피처명 일치하는지 확인하고 실행시키기


In [5]:
# 필수 라이브러리 설치 (Colab에서 한 번 실행)
# !pip install pandas xgboost scikit-learn

#  라이브러리 임포트
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score
from xgboost import XGBClassifier

# ----------------------------------------------------
#  데이터 파일 로드
FILE_PATH = 'hippo_features_clean_icv.csv' # 실제 파일 경로에 맞게 수정해주세요.

try:
    df_raw = pd.read_csv(FILE_PATH)
    print("✅ 데이터 로드 성공. 상위 5개 행:")
    print(df_raw.head().to_markdown(index=False))
    print(f"\n총 데이터 개수: {len(df_raw)}")
except FileNotFoundError:
    print("❌ 오류: 파일을 찾을 수 없습니다. FILE_PATH를 확인해주세요.")
    # 오류 발생 시 이후 코드가 실행되지 않도록 여기서 종료합니다.
    # exit() # Colab에서 전체 스크립트 실행 중단

# ----------------------------------------------------


# ⭐ 1. 변수 정의 순서 수정: 사용 전에 정의되어야 합니다.

LABEL_COL = 'label'

# *주의* APOE4와 ICV 정규화 피처에 결측치가 많아 6개 피처만 사용합니다.
FEATURES_6 = [
    'left_hipp_vol_mm3', 'right_hipp_vol_mm3', 'total_hipp_vol_mm3',
    'asymmetry_index',
    'AGE', 'SEX_FEMALE' # 수정된 임상/유전 위험 인자
]
ALL_COLS_TO_CHECK = FEATURES_6 + [LABEL_COL]


# ⭐ 2. 컬럼명 매핑 및 새로운 피처 생성
df_raw.rename(columns={'age': 'AGE'}, inplace=True)
df_raw['SEX_FEMALE'] = np.where(df_raw['sex'] == 0, 1, 0) # sex=0 (Female)을 1로 매핑
df_raw['label'] = df_raw['dx_y'].map({'CN': 0, 'AD': 1}) # dx_y를 label(0, 1)로 매핑


# ⭐ 3. 결측치 확인 및 제거
initial_count = len(df_raw)
df_clean = df_raw[ALL_COLS_TO_CHECK].copy() # 필요한 컬럼만 선택
df_clean = df_clean.dropna()

removed_count = initial_count - len(df_clean)

if removed_count > 0:
    print(f"\n⚠️ 경고: {removed_count}개의 행(샘플)에 결측치가 있어 제거되었습니다. (총 {len(df_clean)}개 사용)")

df_raw = df_clean # 이제 클린된 데이터프레임을 사용

print("\n✅ 최종 데이터 준비 완료. 클린된 데이터프레임 정보:")
df_raw.info()
# ----------------------------------------------------

# . 데이터 분할 및 XGBoost 모델 학습 (이전 논의된 최종 단계)
X = df_raw[FEATURES_6]
y = df_raw[LABEL_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_model = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n🚀 XGBoost 모델 학습 및 평가 완료.")
print(f"최종 AUC: {roc_auc:.4f}")

✅ 데이터 로드 성공. 상위 5개 행:
| subject_id   |        scan_id | t1_path                                            | mask_path                                                             |   dx_x |   icv_x |   voxel_volume_mm3 |   left_hipp_voxels |   right_hipp_voxels |   left_hipp_vol_mm3 |   right_hipp_vol_mm3 |   total_hipp_vol_mm3 |   asymmetry_index |   left_hipp_vol_icv_norm |   right_hipp_vol_icv_norm |   total_hipp_vol_icv_norm | dx_y   |   age |   sex |       icv_y |
|:-------------|---------------:|:---------------------------------------------------|:----------------------------------------------------------------------|-------:|--------:|-------------------:|-------------------:|--------------------:|--------------------:|---------------------:|---------------------:|------------------:|-------------------------:|--------------------------:|--------------------------:|:-------|------:|------:|------------:|
| 002_S_4213   | 20110902182731 | D:\ADNI\raw_nifti\002_S_4213_20110902182

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [10:51:29] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [28]:
import unittest
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score
from xgboost import XGBClassifier

# ==============================================================================
# I. 유틸리티 함수 정의
# ==============================================================================

def split_data(df, features, target_col, split_ratio, random_state=42):
    """데이터셋을 지정된 비율로 분할합니다."""
    X = df[features]
    y = df[target_col]

    # 70:10:20 분할 로직
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=random_state)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.125, stratify=y_train_val, random_state=random_state)

    print(f"\n[Split: {split_ratio}] Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test


def evaluate_model(model, X_test, y_test, title=""):
    """모델을 평가하고 지표를 계산합니다."""
    # 예측 확률 (y_prob)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    # 지표 계산
    auc = roc_auc_score(y_test, y_prob)
    accuracy = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    specificity = recall_score(y_test, y_pred, pos_label=0)
    precision = precision_score(y_test, y_pred, zero_division=0)

    print(f"\n--- {title} 결과 ---")
    print(f"AUC (Area Under Curve): {auc:.4f}")
    print(f" 정확도 (Accuracy): {accuracy:.4f}")
    print(f"   민감도 (Sensitivity/Recall): {recall:.4f}")
    print(f"   특이도 (Specificity): {specificity:.4f}")
    print(f"   정밀도 (Precision): {precision:.4f}")

    return auc, accuracy

# ==============================================================================
# II. 데이터 준비 및 분할 (변수 정의)
# ==============================================================================

print("--- 🧠 Memora 프로젝트 모델 품질 및 성능 테스트 시작 ---")

# --- 데이터 로드 (Mock Data 생성) ---
# **이 부분을 실제 CSV 파일 로드 코드로 대체해야 합니다.**
np.random.seed(42)
N_SAMPLES = 200
data = {
    'Label': np.random.randint(0, 2, N_SAMPLES),
    'ICV': np.random.rand(N_SAMPLES),
    'Left_Volume': np.random.rand(N_SAMPLES),
    'Right_Volume': np.random.rand(N_SAMPLES),
    'Asymmetry_Index': np.random.rand(N_SAMPLES),
    'Feature_D': np.random.rand(N_SAMPLES),
    'Feature_E': np.random.rand(N_SAMPLES),
    'Feature_F': np.random.rand(N_SAMPLES),
    'Feature_G': np.random.rand(N_SAMPLES),
    'Feature_H': np.random.rand(N_SAMPLES),
    'Feature_I': np.random.rand(N_SAMPLES),
}
# 성능 테스트를 위해 임의로 Label을 두 피처와 연관시킵니다.
data['Label'] = ((data['Left_Volume'] + data['Right_Volume']) > 1.2).astype(int)
df = pd.DataFrame(data)

TARGET_COL = 'Label'
FEATURES_10 = ['ICV', 'Left_Volume', 'Right_Volume', 'Asymmetry_Index',
               'Feature_D', 'Feature_E', 'Feature_F', 'Feature_G',
               'Feature_H', 'Feature_I']

# 70:10:20 분할 수행
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df, FEATURES_10, TARGET_COL, '70:10:20'
)

# 학습+검증 세트 통합 (튜닝에 사용)
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])
print(f"튜닝 데이터셋 크기 (Train + Val): {len(X_train_val)}")


# ==============================================================================
# III. 모델 최적화 (하이퍼파라미터 튜닝) - grid_search 및 best_model 정의
# ==============================================================================

print("\n--- 🔎 하이퍼파라미터 튜닝 시작 (GridSearchCV) ---")

# 튜닝할 파라미터 그리드 정의 (간단한 예시)
param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}

# XGBoost 기본 모델 정의
xgb_base = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# 교차 검증 객체 정의
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV 설정 및 실행
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=0, # 출력 간소화
    n_jobs=-1
)

grid_search.fit(X_train_val, y_train_val)

# 최적 파라미터 및 최고 성능
best_params = grid_search.best_params_
best_score = grid_search.best_score_
best_model = grid_search.best_estimator_ # <-- best_model 정의됨

print(f"\n✅ 튜닝 완료. 최적 파라미터: {best_params}")
print(f"✅ 교차 검증 최고 AUC (Training/Validation): {best_score:.4f}")


# ==============================================================================
# IV. 최종 성능 및 안정성 테스트 (요청하신 부분)
# ==============================================================================

print("\n" + "="*50)
print("--- 🎯 최종 모델 성능 테스트 (최적 모델 & Test Set) ---")
print("="*50)

# 1. 최적 모델을 Test Set에 적용하여 최종 성능 측정
auc_final, acc_final = evaluate_model(
    best_model, X_test, y_test, title="최종 XGBoost 모델 (Test Set)"
)

# 2. 성능 안정성 보고 (교차 검증 결과)
# grid_search 변수를 사용하여 CV 결과에서 최적의 폴드 점수를 가져옵니다.
cv_results = pd.DataFrame(grid_search.cv_results_)
best_cv_run = cv_results[cv_results['params'] == best_params].iloc[0]

# 5-Fold AUC 점수 추출
mean_auc_cv = best_cv_run['mean_test_score']
std_auc_cv = best_cv_run['std_test_score']

print("\n--- 📊 모델 안정성 평가 (5-Fold 교차 검증) ---")
print(f"최적 모델의 평균 AUC: {mean_auc_cv:.4f} (± {std_auc_cv:.4f})")
print(f"모델의 안정성 (AUC 표준편차): {std_auc_cv:.4f}")
print("-" * 50)
print(f"최종 성능 (Test Set AUC): {auc_final:.4f}")
print("--- 🏁 모든 성능 테스트 완료 ---")

# ======================================================================
# V. 🔥 피처 10개 vs 피처 7개 모델 성능 비교
# ======================================================================

print("\n" + "="*60)
print("🔥 피처 10개 모델 vs 7개 모델 성능 비교 시작")
print("="*60)

# -----------------------
# 1) 피처 7개 정의
# -----------------------
FEATURES_7 = [
    'ICV', 'Left_Volume', 'Right_Volume',
    'Asymmetry_Index', 'Feature_D', 'Feature_E', 'Feature_F'
]

# -----------------------
# 2) 7개 피처 데이터 분리
# -----------------------
X_train_7, X_val_7, X_test_7, y_train_7, y_val_7, y_test_7 = split_data(
    df, FEATURES_7, TARGET_COL, '70:10:20 (7 features)'
)

X_train_val_7 = pd.concat([X_train_7, X_val_7])
y_train_val_7 = pd.concat([y_train_7, y_val_7])



grid_search_7 = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=0,
    n_jobs=-1
)

grid_search_7.fit(X_train_val_7, y_train_val_7)

best_params_7 = grid_search_7.best_params_
best_score_7 = grid_search_7.best_score_
best_model_7 = grid_search_7.best_estimator_

print(f"\n[7 Features] 최적 파라미터: {best_params_7}")
print(f"[7 Features] 최고 CV AUC: {best_score_7:.4f}")


# -----------------------
# 최종 성능 측정
# -----------------------
auc_10, acc_10 = auc_final, acc_final   # 기존 10개 피처 모델 결과 사용
auc_7, acc_7 = evaluate_model(best_model_7, X_test_7, y_test_7, title="XGBoost (7 Features)")

# 7개 모델 CV 안정성 정보
cv_results_7 = pd.DataFrame(grid_search_7.cv_results_)
best_cv_run_7 = cv_results_7[cv_results_7['params'] == best_params_7].iloc[0]
mean_auc_cv_7 = best_cv_run_7['mean_test_score']
std_auc_cv_7 = best_cv_run_7['std_test_score']


# -----------------------
# 5) 결과 비교 표 출력
# -----------------------
print("\n" + "-"*65)
print("📊 피처 10개 vs 7개 모델 성능 비교")
print("-"*65)

print(f"{'항목':<25} | {'10개 피처 모델':<20} | {'7개 피처 모델':<20}")
print("-"*65)
print(f"{'평균 AUC':<25} | {mean_auc_cv:.4f}            | {mean_auc_cv_7:.4f}")
print(f"{'표준편차':<25} | {std_auc_cv:.4f}            | {std_auc_cv_7:.4f}")
print(f"{'Test Set AUC':<25} | {auc_10:.4f}               | {auc_7:.4f}")
print(f"{'Test Accuracy':<25} | {acc_10:.4f}               | {acc_7:.4f}")

print("-"*65)
print("🔥 피처 수에 따른 성능 비교 완료")
print("="*65)


--- 🧠 Memora 프로젝트 모델 품질 및 성능 테스트 시작 ---

[Split: 70:10:20] Train=140, Val=20, Test=40
튜닝 데이터셋 크기 (Train + Val): 160

--- 🔎 하이퍼파라미터 튜닝 시작 (GridSearchCV) ---


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:30:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



✅ 튜닝 완료. 최적 파라미터: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
✅ 교차 검증 최고 AUC (Training/Validation): 0.9855

--- 🎯 최종 모델 성능 테스트 (최적 모델 & Test Set) ---

--- 최종 XGBoost 모델 (Test Set) 결과 ---
AUC (Area Under Curve): 1.0000
 정확도 (Accuracy): 0.9750
   민감도 (Sensitivity/Recall): 1.0000
   특이도 (Specificity): 0.9655
   정밀도 (Precision): 0.9167

--- 📊 모델 안정성 평가 (5-Fold 교차 검증) ---
최적 모델의 평균 AUC: 0.9855 (± 0.0133)
모델의 안정성 (AUC 표준편차): 0.0133
--------------------------------------------------
최종 성능 (Test Set AUC): 1.0000
--- 🏁 모든 성능 테스트 완료 ---

🔥 피처 10개 모델 vs 7개 모델 성능 비교 시작

[Split: 70:10:20 (7 features)] Train=140, Val=20, Test=40

[7 Features] 최적 파라미터: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}
[7 Features] 최고 CV AUC: 0.9894

--- XGBoost (7 Features) 결과 ---
AUC (Area Under Curve): 0.9969
 정확도 (Accuracy): 0.9750
   민감도 (Sensitivity/Recall): 1.0000
   특이도 (Specificity): 0.9655
   정밀도 (Precision): 0.9167

----------------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [11:30:59] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [24]:
## 전체 피처 분류 정확도 측정

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=200)


# 1. 전체 피처 학습
model.fit(X_train, y_train)

# 2. 전체 피처 예측
y_pred_full = model.predict(X_test)

# 3. 정확도 계산 및 출력
accuracy_full = accuracy_score(y_test, y_pred_full)


print(f"✅ 전체 피처 분류 정확도: {accuracy_full:.4f}")

✅ 전체 피처 분류 정확도: 0.9500


In [23]:
##일부 피처 분류 정확도 측정

subset_features = X_train.columns[:2].tolist()
X_train_subset = X_train[subset_features]
X_test_subset = X_test[subset_features]

# 1. 새 모델 생성 및 일부 피처 학습
model_subset = LogisticRegression(max_iter=200)
model_subset.fit(X_train_subset, y_train)

# 2. 일부 피처 예측
y_pred_subset = model_subset.predict(X_test_subset)

# 3. 정확도 계산 및 출력
accuracy_subset = accuracy_score(y_test, y_pred_subset)

print(f"✅일부 피처 분류 정확도: {accuracy_subset:.4f}")

✅일부 피처 분류 정확도: 0.7750


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, classification_report
from xgboost import XGBClassifier


MANUAL_THRESHOLD = 0.400
FILE_PATH = 'hippo_features_clean_icv.csv'
LABEL_COL = 'label'

print("--- 1. 데이터 준비 및 클리닝 ---")
try:
    df_raw = pd.read_csv(FILE_PATH)
    df_raw.columns = df_raw.columns.str.strip()


    FEATURES_6 = [
        'left_hipp_vol_mm3', 'right_hipp_vol_mm3', 'total_hipp_vol_mm3',
        'asymmetry_index', 'AGE', 'SEX_FEMALE'
    ]
    ALL_COLS_TO_CHECK = FEATURES_6 + [LABEL_COL]

    df_raw.rename(columns={'age': 'AGE'}, inplace=True)
    df_raw['SEX_FEMALE'] = np.where(df_raw['sex'] == 0, 1, 0)
    df_raw['label'] = df_raw['dx_y'].map({'CN': 0, 'AD': 1})

    df_clean = df_raw[ALL_COLS_TO_CHECK].copy().dropna()

    X = df_clean[FEATURES_6]
    y = df_clean[LABEL_COL]

    # 70:10:20 분할 로직
    X_train_val_tune, X_test, y_train_val_tune, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42)

    print(f"✅ 데이터 준비 완료ㅡ")
except Exception as e:
    print(f"🚨 치명적인 오류: 데이터 로드/준비 실패 - {e}")
    exit()
# ---------------------------------------------------------------------------------


param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}
xgb_base = XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(estimator=xgb_base, param_grid=param_grid, scoring='roc_auc', cv=cv, verbose=0, n_jobs=-1)
grid_search.fit(X_train_val_tune, y_train_val_tune)

best_params = grid_search.best_params_
best_model = grid_search.best_estimator_


# 최적 임계값 함수 정의

def predict_with_threshold(model, X_data, threshold):
    y_pred_proba = model.predict_proba(X_data)[:, 1]
    y_pred = (y_pred_proba >= threshold).astype(int)
    return y_pred, y_pred_proba

def calculate_full_metrics(y_true, y_pred, y_prob, threshold):
    auc = roc_auc_score(y_true, y_prob)
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    precision = precision_score(y_true, y_pred, zero_division=0)

    print(f" Test 성능 ]")
    print(f"Threshold : {threshold:.3f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Sensitivity : {recall:.4f}")
    print(f"Specificity : {specificity:.4f}")
    print(f"Precision : {precision:.4f}")

    print(f"\n[ 분류 리포트 (Manual Threshold) ]")
    print(classification_report(y_true, y_pred, target_names=['CN', 'AD'], zero_division=0))

# ----------------------------------------------------------------
print("\n--- ✅모델 최종 평가 ---")

y_pred_manual, y_prob_manual = predict_with_threshold(
    best_model, X_test, MANUAL_THRESHOLD
)

calculate_full_metrics(
    y_test, y_pred_manual, y_prob_manual, MANUAL_THRESHOLD
)

--- 1. 데이터 준비 및 클리닝 ---
✅ 데이터 준비 완료ㅡ

--- ✅모델 최종 평가 ---
 Test 성능 ]
Threshold : 0.400
Accuracy : 0.7895
Sensitivity : 0.8056
Specificity : 0.7797
Precision : 0.6905

[ 분류 리포트 (Manual Threshold) ]
              precision    recall  f1-score   support

          CN       0.87      0.78      0.82        59
          AD       0.69      0.81      0.74        36

    accuracy                           0.79        95
   macro avg       0.78      0.79      0.78        95
weighted avg       0.80      0.79      0.79        95

